# 🎨 Cartoonify — Unified Interface

Combines **Reimagine** (FLUX.1-Kontext-dev), **Scene** (Depth ControlNet), and **Portrait**
(Canny ControlNet) into a single interface. One story. One button. Three rendering styles.

**Flow:** Write your story → upload a photo → pick a rendering style → click Cartoonify.
Gemini builds the structured prompt silently on generate — no separate Build Prompt step.

| Style | Pipeline | Best for |
|---|---|---|
| **Reimagine** | FLUX.1-Kontext-dev | Creative reinterpretation — scene can change freely |
| **Scene** | FLUX.1-dev + ControlNet Depth | Crowds, architecture — spatial layout preserved |
| **Portrait** | FLUX.1-dev + ControlNet Canny | Specific person must be recognisable |

**VRAM note:** Only one pipeline loads at a time. Switching modes takes ~60–90 s (one-time per switch).

**Requires:** Google Colab Pro → Runtime → Change runtime type → **A100 GPU**

> Accept the [FLUX.1-Kontext-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev)
> on HuggingFace before running — it is gated separately from FLUX.1-dev.

---

## 1 — Confirm A100 GPU

In [ ]:
!nvidia-smi

## 2 — Install Dependencies

In [ ]:
!pip install --quiet diffusers transformers accelerate sentencepiece
!pip install --quiet gradio
!pip install --quiet huggingface_hub
!pip install --quiet google-genai
!pip install --quiet opencv-python-headless

In [ ]:
import os
os.kill(os.getpid(), 9)
print("IGNORE: session crashed. This is intentional — continue to the next cell.")

## 3 — Imports

In [ ]:
import gc
import datetime
import os
import numpy as np
import cv2
import torch
import gradio as gr
import google.genai as genai
import google.genai.types as genai_types
from PIL import Image
from diffusers import (
    FluxKontextPipeline,
    FluxControlNetPipeline,
    FluxControlNetModel,
)
from transformers import pipeline as hf_pipeline

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'cv2    : {cv2.__version__}')
print(f'gradio : {gr.__version__}')
print(f'genai  : {genai.__version__}')

## 4 — Mount Google Drive

In [ ]:
import shutil
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')

## 5 — API Tokens

**Hugging Face** — FLUX.1-dev and FLUX.1-Kontext-dev are both gated models.
Add a Read token to Colab Secrets as `HF_TOKEN`.

**Google Gemini** — Add your key to Colab Secrets as `GOOGLE_API_KEY`
([aistudio.google.com/apikey](https://aistudio.google.com/apikey)).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN       = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

login(token=HF_TOKEN, add_to_git_credential=False)
print('✓ Logged in to Hugging Face')
print('✓ Google API key loaded' if GOOGLE_API_KEY else '⚠️  GOOGLE_API_KEY not found — story-to-prompt will not work')

## 6 — Configuration

Edit this cell to change the LoRA, trigger word, or default rendering mode.
Nothing else needs to change.

In [ ]:
# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_SOURCE     = 'drive'   # 'huggingface' | 'drive'
LORA_HF_REPO    = 'strangerzonehf/Ghibli-Flux-Cartoon-LoRA'
LORA_HF_WEIGHT  = 'Ghibili-Cartoon-Art.safetensors'
LORA_DRIVE_PATH = '/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/gdo_cartoon.safetensors'

# ── Trigger + fallback prompt ──────────────────────────────────────────────────
DEFAULT_TRIGGER = 'gdo_cartoon'
DEFAULT_PROMPT  = 'satirical cartoon illustration, bold outlines, vivid flat colours, expressive exaggerated features'

# ── Gemini ────────────────────────────────────────────────────────────────────
GOOGLE_MODEL = 'gemini-2.5-flash-lite'

# ── Base models ───────────────────────────────────────────────────────────────
BASE_MODEL_KONTEXT = 'black-forest-labs/FLUX.1-Kontext-dev'
BASE_MODEL_FLUX    = 'black-forest-labs/FLUX.1-dev'
CONTROLNET_MODEL   = 'Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0'
DEPTH_MODEL        = 'depth-anything/Depth-Anything-V2-Small-hf'

# ── Defaults per rendering mode ───────────────────────────────────────────────
DEFAULTS = {
    'reimagine': {'guidance': 2.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'scene':     {'guidance': 3.5, 'cn_scale': 0.8, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'portrait':  {'guidance': 3.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
}
DEFAULT_STEPS = 28
DEFAULT_SEED  = 42
DEFAULT_MODE  = 'Reimagine'  # Reimagine | Scene | Portrait

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Default mode  : {DEFAULT_MODE}')
print(f'LoRA source   : {LORA_SOURCE}')
print(f'Output dir    : {OUTPUT_DIR}')

## 7 — Load Models

`load_pipeline(mode)` handles VRAM switching — unloads the current pipeline before loading the new one.
The initial load uses `DEFAULT_MODE` set in cell-config.

⏱️ First run downloads ~24–29 GB depending on mode. Warm cache: ~1 minute.

In [ ]:
active_mode     = None
pipe            = None
controlnet      = None
depth_estimator = None


def _peft_patch():
    import peft.import_utils as _peft_utils
    import peft.tuners.lora.torchao as _peft_torchao
    _orig = _peft_utils.is_torchao_available
    def _safe():
        try: return _orig()
        except ImportError: return False
    _peft_utils.is_torchao_available = _safe
    _peft_torchao.is_torchao_available = _safe


def load_pipeline(mode: str) -> None:
    """Load the pipeline for the given mode, unloading the current one first."""
    global pipe, controlnet, depth_estimator, active_mode

    mode = mode.lower()
    if mode == active_mode:
        print(f'✓ {mode} already loaded.')
        return

    # ── Unload ────────────────────────────────────────────────────────────────
    if active_mode:
        print(f'Unloading {active_mode} pipeline...')
    if pipe is not None:
        del pipe
        pipe = None
    if controlnet is not None:
        del controlnet
        controlnet = None
    if depth_estimator is not None:
        del depth_estimator
        depth_estimator = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total')

    # ── Load ──────────────────────────────────────────────────────────────────
    if mode == 'reimagine':
        print('Loading FLUX.1-Kontext-dev...')
        pipe = FluxKontextPipeline.from_pretrained(
            BASE_MODEL_KONTEXT, torch_dtype=torch.bfloat16
        ).to('cuda')

    elif mode in ('scene', 'portrait'):
        print('Loading ControlNet...')
        controlnet = FluxControlNetModel.from_pretrained(
            CONTROLNET_MODEL, torch_dtype=torch.bfloat16
        )
        print('Loading FLUX.1-dev...')
        pipe = FluxControlNetPipeline.from_pretrained(
            BASE_MODEL_FLUX, controlnet=controlnet, torch_dtype=torch.bfloat16
        ).to('cuda')
        if mode == 'scene':
            print('Loading depth estimator (CPU)...')
            depth_estimator = hf_pipeline(
                'depth-estimation', model=DEPTH_MODEL, device='cpu'
            )
    else:
        raise ValueError(f'Unknown mode: {mode!r}')

    # ── LoRA ──────────────────────────────────────────────────────────────────
    _peft_patch()
    if LORA_SOURCE == 'huggingface':
        pipe.load_lora_weights(LORA_HF_REPO, weight_name=LORA_HF_WEIGHT)
    else:
        pipe.load_lora_weights(LORA_DRIVE_PATH)

    active_mode = mode
    print(f'\u2705 {mode} pipeline ready.')


# ── Initial load ──────────────────────────────────────────────────────────────
load_pipeline(DEFAULT_MODE.lower())

## 8 — Story-to-Prompt (Gemini)

Same system prompt and function as notebooks 02–04.
Called silently inside `cartoonify()` — no separate Build Prompt step.

In [ ]:
GEMINI_SYSTEM_PROMPT = """You are a prompt engineer for a FLUX.1 LoRA fine-tuned on editorial and political cartoons.
Convert the user's story into a structured image generation prompt.

OUTPUT FORMAT — one line only, no explanation, no preamble:
gdo_cartoon <medium>, <technique>, <color>, <mood>, <commentary>, <composition>

RULES:
- Always start with: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration
- Always include: pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight
- Default color: black and white | monochrome | print cartoon | white background
  (use spot color red only if violence or urgency is central to the story)
- Derive mood, commentary, and composition from the story
- Pipe-separated keywords within each layer; comma between layers
- Max 5 keywords per layer
- Composition: estimate figure count, suggest layout, add speech bubble hint if there is dialogue

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | dark humour | critical | bleak, political commentary | scathing | power critique | deadpan | accusatory, two-group layout | standing figure left | queue of figures right | speech bubble top | eye level | wide shot

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, absurdist | dark | pompous | grotesque | confrontational, political condemnation | war metaphor | accusatory | ironic | scathing, large figure dominates upper frame | tiny figures lower third | desk as stage | top-heavy composition | eye level

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | duplicitous | tense | darkly comic, political commentary | diplomatic hypocrisy | sharp | deadpan | allegorical, two figure layout | shadow duality | frontal handshake foreground | fighting silhouettes background | centred composition | eye level"""


def build_prompt_from_story(story: str, trigger_word: str) -> str:
    if not story.strip():
        raise ValueError('Empty story.')
    if not GOOGLE_API_KEY:
        raise ValueError('GOOGLE_API_KEY not set in Colab Secrets.')

    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=GEMINI_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=300,
        ),
    )
    prompt = response.text.strip()

    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'gdo_cartoon':
        prompt = prompt.replace('gdo_cartoon', active_trigger, 1)

    return prompt


print('\u2713 Gemini story-to-prompt ready')

## 9 — Inference

In [ ]:
def extract_canny(pil_image: Image.Image, low: int, high: int) -> Image.Image:
    img_np = np.array(pil_image)
    edges  = cv2.Canny(img_np, low, high)
    edges  = edges[:, :, None]
    edges  = np.concatenate([edges, edges, edges], axis=2)
    return Image.fromarray(edges)


def cartoonify(
    story,
    image,
    mode_label,
    trigger_word,
    guidance_scale,
    num_steps,
    cn_scale,
    cn_end,
    canny_low,
    canny_high,
    seed,
    prompt_override,
):
    if image is None:
        raise gr.Error('Upload a photo first.')

    mode = mode_label.lower()
    log_lines = []

    def log(msg):
        log_lines.append(msg)
        return '\n'.join(log_lines)

    # ── Step 1: Prompt ────────────────────────────────────────────────────────
    if prompt_override.strip():
        prompt = prompt_override.strip()
        yield None, log('\u2713 Using manual prompt override')
    elif story.strip():
        yield None, log('\u29d7 Gemini building your prompt...')
        try:
            prompt = build_prompt_from_story(story, trigger_word)
            yield None, log('\u2713 Prompt ready')
        except Exception as exc:
            yield None, log(f'\u26a0\ufe0f  Gemini failed ({exc}) — using default prompt')
            prompt = DEFAULT_PROMPT
    else:
        prompt = DEFAULT_PROMPT
        yield None, log('\u2713 No story — using default prompt')

    # Ensure trigger present but not duplicated
    trigger = trigger_word.strip()
    if trigger and not prompt.strip().startswith(trigger):
        full_prompt = f'{trigger} {prompt}'.strip()
    else:
        full_prompt = prompt.strip()

    # ── Step 2: Pipeline switch ───────────────────────────────────────────────
    if mode != active_mode:
        yield None, log(f'\u29d7 Switching to {mode_label} pipeline (~60\u201390 s)...')
        load_pipeline(mode)
        yield None, log(f'\u2713 {mode_label} pipeline ready')

    # ── Step 3: Resize ────────────────────────────────────────────────────────
    src = Image.fromarray(image).convert('RGB')
    src = src.resize((1024, 1024), Image.LANCZOS)
    generator = torch.Generator(device='cuda').manual_seed(int(seed))

    # ── Step 4: Inference ─────────────────────────────────────────────────────
    with torch.inference_mode():

        if mode == 'reimagine':
            yield None, log('\u29d7 FLUX Kontext rendering...')
            result = pipe(
                image=src,
                prompt=full_prompt,
                width=1024, height=1024,
                num_inference_steps=int(num_steps),
                guidance_scale=float(guidance_scale),
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
            ).images[0]

        elif mode == 'scene':
            yield None, log('\u29d7 Reading scene depth...')
            depth_out  = depth_estimator(src)
            depth_arr  = np.array(depth_out['depth'])
            d_min, d_max = depth_arr.min(), depth_arr.max()
            if d_max > d_min:
                depth_norm = ((depth_arr - d_min) / (d_max - d_min) * 255).astype(np.uint8)
            else:
                depth_norm = np.zeros_like(depth_arr, dtype=np.uint8)
            control_image = Image.fromarray(depth_norm).convert('RGB')
            yield None, log('\u2713 Depth map ready\n\u29d7 FLUX rendering...')
            result = pipe(
                prompt=full_prompt,
                control_image=control_image,
                control_mode=2,
                width=1024, height=1024,
                num_inference_steps=int(num_steps),
                guidance_scale=float(guidance_scale),
                controlnet_conditioning_scale=float(cn_scale),
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
            ).images[0]

        elif mode == 'portrait':
            yield None, log('\u29d7 Extracting portrait outlines...')
            canny_image = extract_canny(src, int(canny_low), int(canny_high))
            yield None, log('\u2713 Outlines extracted\n\u29d7 FLUX rendering...')
            result = pipe(
                prompt=full_prompt,
                control_image=canny_image,
                control_mode=0,
                width=1024, height=1024,
                num_inference_steps=int(num_steps),
                guidance_scale=float(guidance_scale),
                controlnet_conditioning_scale=float(cn_scale),
                control_guidance_end=float(cn_end),
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
            ).images[0]

    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    suffix = {'reimagine': 'kontext', 'scene': 'depth', 'portrait': 'canny'}[mode]
    save_path = f'{OUTPUT_DIR}/{ts}_cartoonify_{suffix}.png'
    result.save(save_path)

    gc.collect()
    torch.cuda.empty_cache()

    yield result, log('\u2713 Done \u2014 saved to Drive')


print('\u2713 Inference functions ready')

## 10 — Launch Interface

Re-run this cell to restart the UI without reloading models.

**Switching rendering styles** during a session triggers a pipeline swap (~60–90 s).
Within a style, each generation takes ~30–45 s at 28 steps.

In [ ]:
MODE_DESCRIPTIONS = {
    'Reimagine': 'FLUX reads the full image and recomposes it in cartoon style. Best for creative reinterpretation \u2014 scene and staging can change freely.',
    'Scene':     'Depth map preserves the spatial layout and figure positions. Best for crowds, architecture, and compositions where structure matters.',
    'Portrait':  'Canny edges preserve face outlines and silhouettes. Best when a specific person must be immediately recognisable.',
}


def update_mode(mode):
    d = DEFAULTS[mode.lower()]
    is_kontext  = mode == 'Reimagine'
    is_portrait = mode == 'Portrait'
    return (
        MODE_DESCRIPTIONS[mode],
        gr.update(value=d['guidance']),
        gr.update(visible=not is_kontext, value=d['cn_scale']),
        gr.update(visible=is_portrait,    value=d['cn_end']),
        gr.update(visible=is_portrait,    value=d['canny_low']),
        gr.update(visible=is_portrait,    value=d['canny_high']),
    )


CSS = '''
.gradio-container {
    font-family: "Inter", -apple-system, BlinkMacSystemFont, sans-serif !important;
    background: #f0eff5 !important;
    max-width: 1280px !important;
    margin: 0 auto !important;
    padding: 1.5rem !important;
}
#app-header {
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 55%, #24243e 100%);
    border-radius: 18px;
    padding: 2.5rem 2rem 2rem;
    text-align: center;
    margin-bottom: 1.5rem;
    box-shadow: 0 8px 30px rgba(48, 43, 99, 0.35);
}
.app-title {
    font-size: 3rem; font-weight: 900; letter-spacing: -2px;
    color: #ffffff; margin: 0 0 0.5rem; line-height: 1; display: block;
}
.app-subtitle {
    font-size: 1rem; color: #c4b5fd; margin: 0;
    font-weight: 400; line-height: 1.6; display: block;
}
.input-card, .output-card {
    background: #ffffff; border-radius: 16px;
    padding: 1.5rem 1.5rem 1.75rem;
    box-shadow: 0 1px 4px rgba(0,0,0,0.07); border: 1px solid #e5e7eb;
}
.section-label {
    display: block; font-size: 0.68rem; font-weight: 800;
    text-transform: uppercase; letter-spacing: 0.12em;
    color: #7c3aed; margin-bottom: 0.5rem; margin-top: 1.1rem;
}
.section-label:first-child { margin-top: 0; }
.upload-zone {
    border: 2.5px dashed #7c3aed !important; border-radius: 14px !important;
    background: #faf5ff !important; transition: border-color 0.2s, background 0.2s !important;
}
.upload-zone:hover { border-color: #5b21b6 !important; background: #f5edff !important; }
#mode-selector .wrap { display: flex !important; gap: 0.5rem !important; }
#mode-selector label {
    flex: 1 !important; text-align: center !important;
    padding: 0.65rem 0.5rem !important; border: 2px solid #ddd6fe !important;
    border-radius: 10px !important; cursor: pointer !important;
    font-weight: 700 !important; font-size: 0.85rem !important;
    background: #faf5ff !important; color: #5b21b6 !important;
    transition: all 0.2s !important;
}
#mode-selector label:has(input:checked) {
    background: #7c3aed !important; border-color: #7c3aed !important;
    color: white !important;
}
#mode-selector input[type="radio"] { display: none !important; }
.mode-desc p {
    background: #f5f3ff; border: 1px solid #ddd6fe; border-radius: 10px;
    padding: 0.65rem 1rem; font-size: 0.82rem; color: #5b21b6;
    line-height: 1.5; margin: 0.4rem 0 0.4rem;
}
.process-log textarea {
    font-family: "JetBrains Mono", "Fira Code", monospace !important;
    font-size: 0.8rem !important; background: #f9fafb !important;
    border-radius: 10px !important; color: #374151 !important;
}
.output-image { border-radius: 12px !important; overflow: hidden !important; }
#generate-btn {
    background: linear-gradient(135deg, #7c3aed 0%, #5b21b6 100%) !important;
    color: white !important; border: none !important; border-radius: 12px !important;
    font-size: 1.1rem !important; font-weight: 700 !important;
    height: 56px !important; width: 100% !important;
    box-shadow: 0 4px 18px rgba(124,58,237,0.38) !important;
    transition: opacity 0.2s, transform 0.15s !important; margin-top: 1rem !important;
}
#generate-btn:hover  { opacity: 0.90 !important; transform: translateY(-1px) !important; }
#generate-btn:active { transform: translateY(0) !important; }
#generate-btn:disabled { opacity: 0.5 !important; cursor: wait !important; }
footer { display: none !important; }
'''


with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title='Cartoonify') as demo:

    gr.HTML('''
        <div id="app-header">
            <span class="app-title">&#127912; Cartoonify</span>
            <span class="app-subtitle">
                Voice up. Turn your story into a visual, satirical illustration.
            </span>
        </div>
    ''')

    with gr.Row(equal_height=False):

        # ── Left column — Inputs ───────────────────────────────────────────────
        with gr.Column(scale=5, min_width=360, elem_classes=['input-card']):

            gr.HTML('<span class="section-label">&#9312; What\'s the story?</span>')
            story_input = gr.Textbox(
                label='',
                placeholder=(
                    'A politician hands out empty promises while people queue for food...\n'
                    'A general behind a desk of medals while tiny soldiers march across a map below...'
                ),
                lines=5,
                max_lines=10,
            )

            gr.HTML('<span class="section-label">&#9313; Upload a photo</span>')
            img_input = gr.Image(
                label='',
                type='numpy',
                elem_classes=['upload-zone'],
                sources=['upload', 'clipboard'],
                height=220,
            )

            gr.HTML('<span class="section-label">&#9314; Rendering style</span>')
            mode_selector = gr.Radio(
                choices=['Reimagine', 'Scene', 'Portrait'],
                value=DEFAULT_MODE,
                label='',
                elem_id='mode-selector',
            )
            mode_desc = gr.Markdown(
                value=MODE_DESCRIPTIONS[DEFAULT_MODE],
                elem_classes=['mode-desc'],
            )

            with gr.Accordion('\u2699\ufe0f Advanced Settings', open=False):
                guidance_slider = gr.Slider(
                    minimum=1.0, maximum=10.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['guidance'],
                    step=0.5, label='Guidance Scale',
                )
                steps_slider = gr.Slider(
                    minimum=10, maximum=50, value=DEFAULT_STEPS,
                    step=1, label='Inference Steps',
                )
                cn_scale_slider = gr.Slider(
                    minimum=0.1, maximum=1.5,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_scale'],
                    step=0.05, label='ControlNet Scale',
                    visible=DEFAULT_MODE != 'Reimagine',
                )
                cn_end_slider = gr.Slider(
                    minimum=0.3, maximum=1.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_end'],
                    step=0.05, label='ControlNet Guidance End',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                canny_low_slider = gr.Slider(
                    minimum=0, maximum=200,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_low'],
                    step=10, label='Canny Low Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                canny_high_slider = gr.Slider(
                    minimum=50, maximum=500,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_high'],
                    step=10, label='Canny High Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                seed_input = gr.Number(value=DEFAULT_SEED, label='Seed', precision=0)
                trigger_input = gr.Textbox(
                    value=DEFAULT_TRIGGER, label='LoRA Trigger Word', lines=1,
                )
                with gr.Accordion('Edit prompt directly', open=False):
                    prompt_input = gr.Textbox(
                        label='Prompt override (leave empty to use Gemini + story)',
                        placeholder='Leave empty \u2014 Gemini builds the prompt from your story.',
                        lines=3, max_lines=6, value='',
                    )

            generate_btn = gr.Button(
                '\U0001f3a8  Cartoonify  \u2192',
                variant='primary',
                elem_id='generate-btn',
            )

        # ── Right column — Outputs ─────────────────────────────────────────────
        with gr.Column(scale=6, min_width=480, elem_classes=['output-card']):

            gr.HTML('<span class="section-label">Processing</span>')
            log_output = gr.Textbox(
                label='',
                lines=5,
                max_lines=8,
                interactive=False,
                elem_classes=['process-log'],
                placeholder='Click Cartoonify to begin...',
                show_label=False,
            )

            gr.HTML('<span class="section-label">&#127917; Result</span>')
            result_output = gr.Image(
                label='',
                type='pil',
                elem_classes=['output-image'],
                show_download_button=True,
                height=580,
                interactive=False,
            )

    # ── Event wiring ──────────────────────────────────────────────────────────
    mode_selector.change(
        fn=update_mode,
        inputs=[mode_selector],
        outputs=[
            mode_desc,
            guidance_slider,
            cn_scale_slider,
            cn_end_slider,
            canny_low_slider,
            canny_high_slider,
        ],
    )

    generate_btn.click(
        fn=cartoonify,
        inputs=[
            story_input,
            img_input,
            mode_selector,
            trigger_input,
            guidance_slider,
            steps_slider,
            cn_scale_slider,
            cn_end_slider,
            canny_low_slider,
            canny_high_slider,
            seed_input,
            prompt_input,
        ],
        outputs=[result_output, log_output],
        api_name='cartoonify',
    )


demo.launch(share=True, show_error=True, quiet=False)